# 03 - Customer Segmentation (Clustering)

## 11. Preprocessing for Clustering

Features: `recency`, `frequency`, `monetary` (log1p-transformed, then standardized). `CustomerID` is an identifier and the R/F/M scores are derived from the same columns, so neither is used as a feature.

In [1]:
import numpy as np
import pandas as pd
import plotly.express as px
from scipy.stats import skew
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42

rfm = pd.read_parquet("../data/customer_rfm.parquet")
FEATURES = ["recency", "frequency", "monetary"]

print("Customers:", len(rfm))
print("Missing values in features:", rfm[FEATURES].isna().sum().sum())
print("Min values:", rfm[FEATURES].min().to_dict())
print("All features positive:", bool((rfm[FEATURES] > 0).all().all()))
rfm[FEATURES].describe(percentiles=[.5, .9, .99]).round(2)

Customers: 4320
Missing values in features: 0
Min values: {'recency': 1.0, 'frequency': 1.0, 'monetary': 2.9}
All features positive: True


,recency,frequency,monetary
count,4320.00,4320.00,4320.00
mean,93.12,4.26,1914.27
std,100.12,7.65,8303.78
min,1.00,1.00,2.90
50%,51.00,2.00,650.42
90%,264.00,9.00,3451.85
99%,369.00,30.00,17061.55
max,374.00,206.00,278778.02


In [2]:
from sklearn.preprocessing import PowerTransformer, RobustScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

shapes = {
    "log1p": lambda d: np.log1p(d),
    "yeo-johnson": lambda d: pd.DataFrame(
        PowerTransformer(method="yeo-johnson", standardize=False).fit_transform(d),
        columns=d.columns, index=d.index),
}
scalers = {"standard": StandardScaler, "robust": RobustScaler}

rows, spaces = [], {}
for sn, sf in shapes.items():
    shaped = sf(rfm[FEATURES])
    for cn, Cls in scalers.items():
        Z = Cls().fit_transform(shaped)
        km = KMeans(n_clusters=4, n_init=10, random_state=RANDOM_STATE).fit(Z)
        sk = shaped.apply(skew).round(2)
        rows.append({
            "shape": sn, "scaler": cn,
            "skew_recency": sk["recency"], "skew_frequency": sk["frequency"],
            "skew_monetary": sk["monetary"],
            "max_abs_z": round(np.abs(Z).max(), 1),
            "silhouette_k4": round(silhouette_score(Z, km.labels_), 3),
        })
        spaces[(sn, cn)] = Z

cmp_tbl = pd.DataFrame(rows)
print(cmp_tbl.to_string(index=False))

      shape   scaler  skew_recency  skew_frequency  skew_monetary  max_abs_z  silhouette_k4
      log1p standard         -0.32            1.21           0.36        5.8          0.337
      log1p   robust         -0.32            1.21           0.36        3.9          0.334
yeo-johnson standard         -0.05            0.22          -0.01        5.2          0.335
yeo-johnson   robust         -0.05            0.22          -0.01        3.8          0.333


In [3]:
SHAPE, SCALER = "log1p", "standard"   # final choice

shaped = shapes[SHAPE](rfm[FEATURES])
scaler = scalers[SCALER]()
X = pd.DataFrame(scaler.fit_transform(shaped), columns=FEATURES, index=rfm.index)

# Sanity checks
assert X.shape == (len(rfm), 3)
assert not X.isna().any().any()
assert np.allclose(X.mean(), 0, atol=1e-6) and np.allclose(X.std(ddof=0), 1, atol=1e-6)
assert np.allclose(X.values, spaces[(SHAPE, SCALER)])   # matches the compared option

print(f"Final feature matrix: {X.shape} | shape={SHAPE}, scaler={SCALER}")
print("Features:", FEATURES, "| CustomerID and R/F/M scores excluded")
X.head()

Final feature matrix: (4320, 3) | shape=log1p, scaler=standard
Features: ['recency', 'frequency', 'monetary'] | CustomerID and R/F/M scores excluded


,recency,frequency,monetary
0,-1.890528,1.079168,1.451577
1,0.372268,0.390004,0.572918
2,-0.659145,-0.953549,0.584142
3,1.442796,-0.953549,-0.694020
4,-0.168061,1.079168,0.471077


### Section 11 Summary

**Decision 11:** Features are `recency`, `frequency`, and `monetary` for 4,320 customers (all positive, no missing values), transformed with `np.log1p` and standardized with `StandardScaler`. `CustomerID` and the R/F/M scores are excluded (identifier; derived from the same columns).

**Options compared:** log1p vs Yeo-Johnson, and StandardScaler vs RobustScaler. Silhouette (K-Means, K=4) ranged only from 0.330 to 0.337, so no option separated segments meaningfully better. Yeo-Johnson reduced the skew of `frequency` (0.22 vs 1.21) and RobustScaler reduced the maximum absolute z-score (3.9 vs 5.8), but neither improved separation.

**Why `log1p` + `StandardScaler`:** simplest to explain (log differences are percentage changes), and extreme customers are handled separately by DBSCAN in Section 12.

**Final matrix:** `X` (4,320 x 3), mean 0 and std 1 per feature (verified with assertions), used in Section 12.

**Limitation:** `frequency` keeps a residual skew of 1.21 after log1p; the silhouette comparison across feature spaces is an indicator, not a formal test.

## 12. Clustering & Evaluation

### 12.1 Choosing K for K-Means

K is chosen from the Elbow (inertia) and Silhouette together, with Davies-Bouldin and the smallest cluster size as additional checks, not from the elbow alone.

In [4]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score
import plotly.graph_objects as go
from plotly.subplots import make_subplots

rows = []
for k in range(2, 11):
    km = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE).fit(X)
    rows.append({
        "k": k,
        "inertia": km.inertia_,
        "silhouette": silhouette_score(X, km.labels_),
        "davies_bouldin": davies_bouldin_score(X, km.labels_),
        "smallest_cluster": int(np.bincount(km.labels_).min()),
    })
k_tbl = pd.DataFrame(rows)
print(k_tbl.round(3).to_string(index=False))

fig = make_subplots(rows=1, cols=3, subplot_titles=(
    "Elbow (inertia, lower is tighter)",
    "Silhouette (higher is better)",
    "Davies-Bouldin (lower is better)"))
fig.add_trace(go.Scatter(x=k_tbl["k"], y=k_tbl["inertia"], mode="lines+markers"), 1, 1)
fig.add_trace(go.Scatter(x=k_tbl["k"], y=k_tbl["silhouette"], mode="lines+markers"), 1, 2)
fig.add_trace(go.Scatter(x=k_tbl["k"], y=k_tbl["davies_bouldin"], mode="lines+markers"), 1, 3)
fig.update_xaxes(title_text="K", dtick=1)
fig.update_layout(showlegend=False, title="K selection for K-Means")
fig.show()

 k  inertia  silhouette  davies_bouldin  smallest_cluster
 2 6440.957       0.433           0.890              1654
 3 4818.719       0.339           1.038               746
 4 3892.990       0.337           1.019               692
 5 3241.279       0.318           0.982               359
 6 2814.595       0.313           1.018               342
 7 2507.419       0.302           0.990               191
 8 2304.683       0.278           1.010               210
 9 2129.384       0.291           1.021               106
10 1965.118       0.276           1.022                98


In [5]:
from sklearn.metrics import adjusted_rand_score, silhouette_score, davies_bouldin_score

rows = []
for k in range(2, 8):
    base = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE).fit(X)
    aris = [adjusted_rand_score(
                base.labels_,
                KMeans(n_clusters=k, n_init=10, random_state=s).fit(X).labels_)
            for s in range(1, 8)]
    tmp = rfm.assign(c=base.labels_)
    g = tmp.groupby("c").agg(n=("monetary", "size"),
                             rec=("recency", "median"),
                             rev=("monetary", "sum"))
    g["share"] = g["rev"] / g["rev"].sum()
    rows.append({
        "k": k,
        "silhouette": silhouette_score(X, base.labels_),
        "davies_bouldin": davies_bouldin_score(X, base.labels_),
        "stability_ARI": np.mean(aris),
        "smallest_cluster": int(g["n"].min()),
        "top_share": g["share"].max(),
        "recency_range_days": g["rec"].max() - g["rec"].min(),
    })

cmp_k = pd.DataFrame(rows)
print(cmp_k.round(3).to_string(index=False))

for k in (2, 3, 4, 5, 6):
    km = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE).fit(X)
    p = rfm.assign(c=km.labels_).groupby("c").agg(
        n=("monetary", "size"),
        recency=("recency", "median"),
        frequency=("frequency", "median"),
        monetary=("monetary", "median"),
        rev=("monetary", "sum"))
    p["revenue_share"] = p["rev"] / p["rev"].sum()
    print(f"\nK={k}")
    print(p.drop(columns="rev").sort_values("recency").round(2).to_string())

 k  silhouette  davies_bouldin  stability_ARI  smallest_cluster  top_share  recency_range_days
 2       0.433           0.890          0.999              1654      0.844                80.0
 3       0.339           1.038          0.992               746      0.675               149.0
 4       0.337           1.019          0.951               692      0.642               174.0
 5       0.318           0.982          0.988               359      0.534               211.0
 6       0.313           1.018          0.949               342      0.513               235.0
 7       0.302           0.990          0.960               191      0.439               241.0

K=2
      n  recency  frequency  monetary  revenue_share
c                                                   
1  1654     17.0        6.0   2002.36           0.84
0  2666     97.0        1.0    351.33           0.16

K=3
      n  recency  frequency  monetary  revenue_share
c                                                   
1   746

In [6]:
km4 = KMeans(n_clusters=4, n_init=10, random_state=RANDOM_STATE).fit(X)
km5 = KMeans(n_clusters=5, n_init=10, random_state=RANDOM_STATE).fit(X)
print(pd.crosstab(km4.labels_, km5.labels_, normalize="index").round(2).to_string())

col_0     0     1     2     3     4
row_0                              
0      0.17  0.00  0.13  0.71  0.00
1      0.03  0.00  0.76  0.00  0.20
2      0.60  0.00  0.00  0.00  0.40
3      0.00  0.52  0.00  0.00  0.48


**Direct evidence (K=4 vs K=5):** a cross-tabulation shows that K=4 cluster 2 (median recency 53 days) splits 60% / 40% into a cooling-regulars cluster and an active-regulars cluster in K=5, and that the K=4 top cluster (692 customers) splits roughly 52% / 48% into the champions cluster and active regulars. K=5 therefore separates groups that need different marketing actions (win-back vs cross-selling) that K=4 merges.

In [7]:
K = 5
km_final = KMeans(n_clusters=K, n_init=10, random_state=RANDOM_STATE).fit(X)
rfm["kmeans"] = km_final.labels_

print("Customers per cluster:")
print(rfm["kmeans"].value_counts().sort_index().to_string())
print("\nSilhouette:", round(silhouette_score(X, km_final.labels_), 3))
print("Inertia:", round(km_final.inertia_, 1))
assert rfm["kmeans"].notna().all() and rfm["kmeans"].nunique() == K

Customers per cluster:
kmeans
0     997
1     359
2     871
3    1120
4     973

Silhouette: 0.318
Inertia: 3241.3


### 12 Final K-Means (K = 5)

K-Means was fitted once with K=5 (`n_init=10`, `random_state=42`) on the scaled matrix `X`, and the cluster label was stored in `rfm["kmeans"]`.

| Cluster | Customers |
|---|---|
| 0 | 997 |
| 1 | 359 |
| 2 | 871 |
| 3 | 1,120 |
| 4 | 973 |

Silhouette = 0.318, inertia = 3,241.3. The sizes match the exploratory runs in 12.1, confirming the result is reproducible.

### 12.3 Hierarchical clustering (sample)

Hierarchical clustering (Ward linkage) is run on a random sample of 800 customers because the full distance matrix is heavy and a dendrogram of 4,320 leaves is unreadable. The result is compared with K-Means on the same sample.

In [8]:
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
import plotly.figure_factory as ff

N_SAMPLE = 800
idx = rfm.sample(N_SAMPLE, random_state=RANDOM_STATE).index
Xs = X.loc[idx]

Z = linkage(Xs, method="ward")
hier = fcluster(Z, t=5, criterion="maxclust") - 1   # 5 clusters, labels 0-4

cmp = pd.DataFrame({"hier": hier, "kmeans": rfm.loc[idx, "kmeans"].values}, index=idx)
print("Hierarchical cluster sizes:", np.bincount(hier).tolist())
print("Silhouette (hierarchical):", round(silhouette_score(Xs, hier), 3))
print("Silhouette (K-Means, same sample):", round(silhouette_score(Xs, cmp["kmeans"]), 3))
print("ARI (hierarchical vs K-Means):", round(adjusted_rand_score(hier, cmp["kmeans"]), 3))

print("\nHow each hierarchical cluster maps to K-Means (row %):")
print(pd.crosstab(cmp["hier"], cmp["kmeans"], normalize="index").round(2).to_string())

fig = ff.create_dendrogram(
    Xs.values, linkagefun=lambda x: linkage(x, "ward"),
    color_threshold=Z[-4, 2],
)
fig.update_layout(title="Dendrogram (Ward, 800-customer sample)", width=1000, height=500)
fig.update_xaxes(showticklabels=False)
fig.show()

# Profile of hierarchical clusters (original units, median)
print("\nHierarchical profile (median, sample):")
print(rfm.loc[idx].assign(h=hier).groupby("h").agg(
    n=("monetary", "size"), recency=("recency", "median"),
    frequency=("frequency", "median"), monetary=("monetary", "median")).round(1).to_string())

Hierarchical cluster sizes: [117, 186, 233, 132, 132]
Silhouette (hierarchical): 0.253
Silhouette (K-Means, same sample): 0.308
ARI (hierarchical vs K-Means): 0.476

How each hierarchical cluster maps to K-Means (row %):
kmeans     0     1     2     3     4
hier                                
0       0.00  0.56  0.00  0.00  0.44
1       0.04  0.00  0.02  0.94  0.00
2       0.27  0.00  0.60  0.12  0.01
3       0.79  0.00  0.00  0.02  0.20
4       0.14  0.00  0.20  0.00  0.66



Hierarchical profile (median, sample):
     n  recency  frequency  monetary
h                                   
0  117     11.0       11.0    3542.5
1  186    219.0        1.0     196.7
2  233     41.0        1.0     426.6
3  132     96.0        4.0    1340.3
4  132     18.0        3.0    1196.3


### 12.3 Hierarchical clustering: results

Ward linkage on a random sample of 800 customers, cut at 5 clusters.

| Metric | Hierarchical | K-Means (same sample) |
|---|---|---|
| Silhouette | 0.253 | 0.308 |
| ARI between the two | 0.476 | |

The two methods agree on the extremes: 94% of the hierarchical cluster of lost one-timers (median recency 219 days, 1 order) fall in K-Means cluster 3, and 79% of the cooling-regulars cluster (recency 96, 4 orders) fall in K-Means cluster 0. They disagree on the active customers: the hierarchical method merges champions and most active regulars into one cluster (median recency 11, 11 orders, 3,543 GBP), and splits the rest differently. This is consistent with the boundary between champions and active regulars being soft.

K-Means has the higher silhouette on this sample and scales to all 4,320 customers, so it remains the base segmentation. Limitation: a single 800-customer sample with a fixed random state, so figures are approximate.

In [9]:
heights = Z[-10:, 2][::-1]
gaps = heights[:-1] - heights[1:]
print(pd.DataFrame({
    "clusters_if_cut_below": range(2, 11),
    "merge_height": heights[:9].round(1),
    "drop_to_next": gaps.round(1),
}).to_string(index=False))

 clusters_if_cut_below  merge_height  drop_to_next
                     2          42.1          13.5
                     3          28.6           9.2
                     4          19.4           3.2
                     5          16.2           4.2
                     6          12.0           0.1
                     7          11.9           2.5
                     8           9.4           0.0
                     9           9.4           0.9
                    10           8.5           0.6


**Tree heights:** the largest drops in merge height occur when going from 2 to 3 clusters (13.5) and from 3 to 4 (9.2 after the 3-cluster cut). Beyond 4 the drops are small (3.2 at 4 clusters, 4.2 at 5, then near zero between 6 and 9), so the tree does not point to a single natural number of clusters; it favors a coarse split (2-3) and gives only weak support for 4-5. Together with the silhouette and Davies-Bouldin results, this indicates that the customer base is a continuum rather than clearly separated groups, and K=5 was chosen for marketing usefulness (separate actions per segment), not because the data shows five natural clusters.

### 12.4 DBSCAN (outlier isolation)

DBSCAN is used to isolate customers that do not belong to any dense region. `min_samples = 6` (2 x number of features) and `eps` is chosen from the k-distance plot. Features are the same scaled matrix `X`.

In [10]:
from sklearn.cluster import DBSCAN
from sklearn.neighbors import NearestNeighbors
import plotly.graph_objects as go

MIN_SAMPLES = 6

# k-distance plot: distance of each customer to its 6th nearest neighbour
nn = NearestNeighbors(n_neighbors=MIN_SAMPLES).fit(X)
kdist = np.sort(nn.kneighbors(X)[0][:, -1])

fig = go.Figure(go.Scatter(y=kdist, mode="lines"))
fig.update_layout(title="k-distance plot (k=6), sorted",
                  xaxis_title="Customers (sorted)", yaxis_title="Distance to 6th neighbour")
fig.show()

print("k-distance percentiles:",
      {p: round(float(np.percentile(kdist, p)), 3) for p in (50, 90, 95, 98, 99, 99.5)})

# Sensitivity: how many customers are flagged as noise for several eps values
rows = []
for eps in (0.2, 0.3, 0.4, 0.5, 0.6, 0.8):
    lab = DBSCAN(eps=eps, min_samples=MIN_SAMPLES).fit_predict(X)
    rows.append({"eps": eps,
                 "clusters": len(set(lab)) - (1 if -1 in lab else 0),
                 "noise": int((lab == -1).sum()),
                 "noise_pct": round((lab == -1).mean() * 100, 1)})
print(pd.DataFrame(rows).to_string(index=False))

k-distance percentiles: {50: 0.123, 90: 0.306, 95: 0.4, 98: 0.568, 99: 0.696, 99.5: 0.99}
 eps  clusters  noise  noise_pct
 0.2        26    748       17.3
 0.3         5    282        6.5
 0.4         3    120        2.8
 0.5         2     60        1.4
 0.6         2     33        0.8
 0.8         1     20        0.5


### 12.4b eps = 0.5 vs eps = 0.6

The 27 extra customers flagged at eps = 0.5 (but not at 0.6) are inspected in original units to decide how wide the outlier filter should be.

In [11]:
lab05 = DBSCAN(eps=0.5, min_samples=MIN_SAMPLES).fit_predict(X)
lab06 = DBSCAN(eps=0.6, min_samples=MIN_SAMPLES).fit_predict(X)

tmp = rfm.assign(out05=lab05 == -1, out06=lab06 == -1)
both = tmp[tmp["out05"] & tmp["out06"]]
extra = tmp[tmp["out05"] & ~tmp["out06"]]
only06 = tmp[~tmp["out05"] & tmp["out06"]]

print(f"Noise at 0.5: {tmp['out05'].sum()} | at 0.6: {tmp['out06'].sum()}")
print(f"In both: {len(both)} | only at 0.5 (extra): {len(extra)} | only at 0.6: {len(only06)}")

cols = ["recency", "frequency", "monetary"]
summary = pd.DataFrame({
    "both (core outliers)": both[cols].median(),
    "extra at 0.5": extra[cols].median(),
    "all other customers": tmp[~tmp["out05"]][cols].median(),
}).round(1)
print("\nMedians (original units):")
print(summary.to_string())

print("\nRevenue share:")
for name, d in [("both", both), ("extra at 0.5", extra)]:
    print(f"  {name}: {d['monetary'].sum() / rfm['monetary'].sum():.1%}")

print("\nK-Means cluster of the extra 27:")
print(extra["kmeans"].value_counts().sort_index().to_string())

print("\nExtra customers (original units):")
print(extra.sort_values("monetary", ascending=False)[
    ["CustomerID", "recency", "frequency", "monetary", "is_uk", "kmeans"]
].round(1).to_string(index=False))

Noise at 0.5: 60 | at 0.6: 33
In both: 33 | only at 0.5 (extra): 27 | only at 0.6: 0

Medians (original units):
           both (core outliers)  extra at 0.5  all other customers
recency                     5.0          12.0                 52.0
frequency                  28.0          15.0                  2.0
monetary                30175.8       13375.9                643.2

Revenue share:
  both: 21.5%
  extra at 0.5: 7.1%

K-Means cluster of the extra 27:
kmeans
0     4
1    17
2     2
4     4

Extra customers (original units):
CustomerID  recency  frequency  monetary  is_uk  kmeans
     13694        4         50   62961.5      1       1
     15061        4         48   54250.3      1       1
     17949        2         44   53215.7      1       1
     15769        8         26   51823.7      1       1
     14298        9         44   50862.4      1       1
     13798        2         57   36352.9      1       1
     12931       22         15   33544.8      1       1
     16013   

### 12.4b Result: eps = 0.5 vs eps = 0.6

At eps = 0.5, DBSCAN flags 60 customers (1.4%); at eps = 0.6, it flags 33 (0.8%). Every customer flagged at 0.6 is also flagged at 0.5, so 0.5 is a wider version of the same filter.

| Group | Customers | Median recency | Median frequency | Median monetary (GBP) | Revenue share |
|---|---|---|---|---|---|
| Core outliers (both) | 33 | 5 | 28 | 30,176 | 21.5% |
| Extra at 0.5 only | 27 | 12 | 15 | 13,376 | 7.1% |
| All other customers | 4,260 | 52 | 2 | 643 | n/a |

The 27 extra customers are not ordinary customers on the edge: their median spend is about 20 times that of the rest, and 17 of them already belong to K-Means cluster 1 (the smallest, highest-value cluster). A few (6 customers, from 37 to 401 GBP) are flagged because of an unusual combination of values, not because of high spend.

**Decision 12.4:** `eps = 0.6` (33 customers) is kept as the outlier list ("key accounts"), because the purpose of DBSCAN here is to isolate unusual accounts, while the additional high-value customers are already captured by K-Means cluster 1. The choice is a judgment: the k-distance plot shows no single sharp elbow, and eps = 0.5 would be an equally defensible, wider filter.

### 12.4c Final DBSCAN (eps = 0.6, min_samples = 6)

DBSCAN is fitted once on the scaled matrix `X`. Customers labelled -1 are stored as outliers ("key accounts") and profiled separately in Section 13.

In [12]:
EPS = 0.6

db = DBSCAN(eps=EPS, min_samples=MIN_SAMPLES).fit(X)
rfm["dbscan"] = db.labels_
rfm["is_outlier"] = rfm["dbscan"] == -1

print("DBSCAN clusters (excluding noise):", len(set(db.labels_)) - 1)
print("Outliers:", int(rfm["is_outlier"].sum()), f"({rfm['is_outlier'].mean():.1%})")
print("Outlier revenue share:",
      f"{rfm.loc[rfm['is_outlier'], 'monetary'].sum() / rfm['monetary'].sum():.1%}")

print("\nOutliers by K-Means cluster:")
print(rfm.loc[rfm["is_outlier"], "kmeans"].value_counts().sort_index().to_string())

print("\nOutlier medians (original units):")
print(rfm.loc[rfm["is_outlier"], ["recency", "frequency", "monetary"]].median().round(1).to_string())

assert int(rfm["is_outlier"].sum()) == 33

DBSCAN clusters (excluding noise): 2
Outliers: 33 (0.8%)
Outlier revenue share: 21.5%

Outliers by K-Means cluster:
kmeans
0     3
1    21
2     2
3     5
4     2

Outlier medians (original units):
recency          5.0
frequency       28.0
monetary     30175.8


In [13]:
# 1) Comparison table
mask = ~rfm["is_outlier"]
rows = [
    {"method": "K-Means (K=5)", "clusters": 5,
     "silhouette": silhouette_score(X, rfm["kmeans"]),
     "note": "all 4,320 customers"},
    {"method": "Hierarchical (Ward, K=5)", "clusters": 5,
     "silhouette": silhouette_score(Xs, hier),
     "note": "800-customer sample only"},
]
if rfm.loc[mask, "dbscan"].nunique() > 1:
    rows.append({"method": "DBSCAN (eps=0.6)",
                 "clusters": rfm.loc[mask, "dbscan"].nunique(),
                 "silhouette": silhouette_score(X[mask], rfm.loc[mask, "dbscan"]),
                 "note": "33 outliers excluded"})
else:
    rows.append({"method": "DBSCAN (eps=0.6)", "clusters": 1,
                 "silhouette": np.nan,
                 "note": "one dense cluster + 33 outliers"})
print(pd.DataFrame(rows).round(3).to_string(index=False))

# 2) Interactive 3D view in log space, outliers highlighted
view = rfm.assign(
    log_recency=np.log1p(rfm["recency"]),
    log_frequency=np.log1p(rfm["frequency"]),
    log_monetary=np.log1p(rfm["monetary"]),
    group=np.where(rfm["is_outlier"], "Outlier (DBSCAN)", "K-Means " + rfm["kmeans"].astype(str)),
)
fig = px.scatter_3d(view, x="log_recency", y="log_frequency", z="log_monetary",
                    color="group", hover_name="CustomerID",
                    hover_data=["recency", "frequency", "monetary"], opacity=0.7,
                    title="Customers in log space: K-Means clusters and DBSCAN outliers")
fig.update_traces(marker_size=3)
fig.show()

                  method  clusters  silhouette                     note
           K-Means (K=5)         5       0.318      all 4,320 customers
Hierarchical (Ward, K=5)         5       0.253 800-customer sample only
        DBSCAN (eps=0.6)         2       0.572     33 outliers excluded


### 12.4c Result: final DBSCAN and comparison

DBSCAN (eps = 0.6, min_samples = 6) finds 2 dense clusters and flags 33 customers (0.8%) as outliers. These customers generate 21.5% of net revenue, with median frequency 28 and median monetary 30,176 GBP (versus 2 and 643 GBP for the rest). 21 of them belong to K-Means cluster 1.

| Method | Clusters | Silhouette | Note |
|---|---|---|---|
| K-Means (K=5) | 5 | 0.318 | all 4,320 customers |
| Hierarchical (Ward, K=5) | 5 | 0.253 | 800-customer sample |
| DBSCAN (eps=0.6) | 2 | 0.572 | outliers excluded |

The DBSCAN silhouette is not comparable with the others: it measures separation between only two clusters. It indicates that the non-outlier customers form one large connected region, consistent with the continuum seen in the dendrogram. K-Means remains the base segmentation; DBSCAN is used to isolate key accounts.

## 13. Segment Profiling

Each K-Means cluster is profiled in original units (median recency, frequency, monetary), with size and revenue share. The 33 DBSCAN outliers are profiled separately as key accounts.

In [14]:
prof = rfm.groupby("kmeans").agg(
    customers=("CustomerID", "size"),
    recency=("recency", "median"),
    frequency=("frequency", "median"),
    monetary=("monetary", "median"),
    avg_order_value=("avg_order_value", "median"),
    cancelled_value=("cancelled_value", "median"),
    uk_share=("is_uk", "mean"),
    revenue=("monetary", "sum"),
    outliers=("is_outlier", "sum"),
)
prof["customer_share"] = prof["customers"] / prof["customers"].sum()
prof["revenue_share"] = prof["revenue"] / prof["revenue"].sum()
prof = prof.drop(columns="revenue").sort_values("recency")
print(prof.round(3).to_string())

# Key accounts (DBSCAN outliers)
ka = rfm[rfm["is_outlier"]]
print("\nKey accounts:", len(ka),
      f"| revenue share: {ka['monetary'].sum() / rfm['monetary'].sum():.1%}")
print(ka[["recency", "frequency", "monetary", "avg_order_value"]].median().round(1).to_string())

# Profile without the outliers, to see the clusters' typical customer
reg = rfm[~rfm["is_outlier"]]
print("\nMedians excluding outliers:")
print(reg.groupby("kmeans")[["recency", "frequency", "monetary"]].median().round(1).to_string())

# Interactive comparison
fig = px.bar(prof.reset_index(), x="kmeans", y="revenue_share",
             hover_data=["customers", "recency", "frequency", "monetary"],
             title="Revenue share by K-Means cluster",
             labels={"kmeans": "K-Means cluster", "revenue_share": "Revenue share"})
fig.update_yaxes(tickformat=".0%")
fig.update_xaxes(type="category")
fig.show()

        customers  recency  frequency  monetary  avg_order_value  cancelled_value  uk_share  outliers  customer_share  revenue_share
kmeans                                                                                                                              
1             359      8.0       15.0   5656.75          390.504            58.33     0.889        21           0.083          0.534
4             973     17.0        5.0   1680.76          322.720             4.25     0.903         2           0.225          0.237
2             871     26.0        1.0    332.64          215.080             0.00     0.924         2           0.202          0.038
0             997     85.0        3.0    954.09          356.250             0.00     0.874         3           0.231          0.155
3            1120    219.0        1.0    232.18          197.690             0.00     0.921         5           0.259          0.035

Key accounts: 33 | revenue share: 21.5%
recency                5.0
f

### 13 Results: segment profiles

| Cluster | Name (data-based) | Customers | Median recency (days) | Median frequency | Median monetary (GBP) | Customer share | Revenue share |
|---|---|---|---|---|---|---|---|
| 1 | Champions | 359 | 8 | 15 | 5,657 | 8.3% | 53.4% |
| 4 | Active Regulars | 973 | 17 | 5 | 1,681 | 22.5% | 23.7% |
| 2 | Recent One-timers | 871 | 26 | 1 | 333 | 20.2% | 3.8% |
| 0 | Cooling Regulars | 997 | 85 | 3 | 954 | 23.1% | 15.5% |
| 3 | Lost One-timers | 1,120 | 219 | 1 | 232 | 25.9% | 3.5% |

Names are labels derived from the medians above, not predefined templates.

**Key findings:**
- Cluster 1 (8.3% of customers) generates 53.4% of net revenue; cluster 4 adds 23.7%. Together they are 30.8% of customers and 77.1% of revenue.
- Clusters 2 and 3 both consist of single-order customers (median frequency 1) and differ mainly in recency (26 vs 219 days). Together they are 46.1% of customers but only 7.3% of revenue.
- Cluster 0 customers ordered about 3 times but have been inactive for a median of 85 days, well beyond the typical 22-day gap between orders of repeat buyers (Section 6).
- The 33 DBSCAN key accounts (median frequency 28, median 30,176 GBP) generate 21.5% of revenue; 21 of them are in cluster 1. Excluding them changes cluster medians only slightly.
- UK share is 87-92% in every cluster, so segments are not driven by country.

These are descriptive profiles; they do not explain why customers behave this way.

## 14. Business Recommendations

Each recommendation is tied to a number from the segment profile in Section 13. Actions are suggestions to test, not proven effects.

| Cluster | Segment | Evidence | Suggested action | How to measure |
|---|---|---|---|---|
| 1 | Champions | 359 customers (8.3%), 53.4% of revenue, median recency 8 days, 15 orders | Loyalty perks, early access to new products, referral program. Assign account management to the 33 key accounts (21.5% of revenue) | Retention of cluster 1 customers; share of revenue from this group |
| 4 | Active Regulars | 973 customers (22.5%), 23.7% of revenue, median 5 orders, recency 17 days | Cross-sell related product ranges and bundles; offer volume incentives to move them toward cluster 1 | Share moving to higher frequency; average order value |
| 2 | Recent One-timers | 871 customers (20.2%), 1 order, last purchase 26 days ago, median spend 333 GBP | Welcome sequence aimed at a second purchase within the typical 22-day reorder rhythm | Share placing a second order within 30 to 60 days |
| 0 | Cooling Regulars | 997 customers (23.1%), 15.5% of revenue, 3 orders, inactive for 85 days (vs 22-day typical gap) | Win-back email with a targeted offer, prioritizing customers with the highest past spend | Reactivation rate; revenue recovered |
| 3 | Lost One-timers | 1,120 customers (25.9%), 3.5% of revenue, 1 order, inactive for 219 days | Low-cost, one-off reactivation attempt; do not spend heavily | Response rate versus cost |

### Priorities

1. **Protect** clusters 1 and 4: together 30.8% of customers and 77.1% of revenue.
2. **Recover** cluster 0: the largest group with proven repeat behavior that has gone quiet.
3. **Convert** cluster 2 before it drifts into cluster 3: both are single-order customers and differ mainly in recency (26 vs 219 days).
4. **Limit spend** on cluster 3: 25.9% of customers but only 3.5% of revenue.

### Comparison with the standard RFM segments

Clusters 1, 4, 2, and 0 correspond roughly to Champions, Loyal, New Inflow, and At-Risk. Cluster 0 is not "high-value" (median 954 GBP versus 5,657 for Champions), so it is labeled Cooling Regulars. Cluster 3 has no counterpart in the standard template and was found by the data.

*Recommendations are hypotheses from descriptive data; they should be validated with A/B tests before scaling.*

## 15. Limitations & Next Steps

### Limitations

| # | Limitation | Evidence | Effect on conclusions |
|---|---|---|---|
| 1 | Guest checkouts excluded from customer analysis | 135,037 raw rows (about 25%) had no `CustomerID`; they are kept in sales analysis but not in RFM or clustering | Segments describe registered customers only, not all sales |
| 2 | Only one year of data | Data runs from 2010-12-01 to 2011-12-09 | Seasonality cannot be separated from growth or promotions; Q4 2010 is not available for comparison |
| 3 | December 2011 is incomplete | Data ends on Dec 9 | Recency and December comparisons are affected |
| 4 | Cancellations may refer to purchases before the dataset starts | 85.3% of cancellations with a known customer were matched to an earlier purchase (Section 4.5b); the rest are unmatched | Cancellation rates and some customer net values are approximate (assumption, not verified) |
| 5 | Customers removed from RFM | 13 customers with net monetary <= 0 (returned their purchases or net negative) | Effect is negligible (-237 GBP in total) but they are not segmented |
| 6 | Segments are not sharply separated | Silhouette 0.318 for K-Means; dendrogram shows no single natural number of clusters | K=5 was chosen for marketing usefulness; customers near a boundary could fall in either segment |
| 7 | RFM scoring has known weaknesses | `qcut` on ranks splits Frequency ties; M=5 is very wide (Section 9) | RFM scores are used for description and comparison, not as final segments |
| 8 | No customer-type or reason fields | The data has no wholesale/retail flag and no return reasons | "Wholesale-like" is an order-size indicator, and reasons for returns are unknown |
| 9 | Descriptive data only | No experiments or controls | Findings do not show causes; recommendations are hypotheses |
| 10 | Judgment choices | Dead-stock definition (top 20% by revenue), DBSCAN `eps = 0.6`, `min_samples = 6`, K=5 | Different choices would change counts, though documented alternatives gave similar pictures |

### Next Steps

1. **A/B test the recommendations** from Section 14 (welcome sequence, win-back, loyalty perks) and measure the metrics in the "How to measure" column.
2. **Collect more history** (at least two full years) to separate seasonality from growth and to compare Q4 across years.
3. **Capture customer identity at checkout** to reduce the share of guest rows.
4. **Add customer-type and return-reason fields** to confirm wholesale behavior and understand cancellations.
5. **Track segment movement over time** (for example, monthly transitions between clusters 2, 0, and 3) to see who is drifting away.
6. **Consider a predictive churn or lifetime-value model** once enough history exists.

## 16. Final Conclusion

### The story

**1. The business is seasonal and concentrated.** Net revenue in Oct-Nov 2011 was about 1.96x the Jan-Aug average, and the rise starts in September. The Q4 increase comes from more orders, not larger orders. Revenue is also concentrated: the top 10% of invoices generate 47.6% of revenue, and the top 1% of customers generate 30.5%.

**2. The data was made trustworthy before analysis.** Starting from 541,909 rows, 10,674 (2.0%) were removed or isolated with documented reasons (duplicates, stock adjustments, zero or negative prices, non-product codes, two fully reversed orders). Cancellations were kept and netted against revenue. Guest checkouts (about 25% of rows) stay in sales analysis but are excluded from customer analysis.

**3. Customers are mostly low-frequency, with a small group carrying the revenue.** Among customers with sales, the median is 2 orders, about one third bought only once, and repeat buyers reorder about every 22 days. RFM scores confirm the concentration: the `555` profile (8.0% of customers) generates 45.7% of net revenue.

**4. Clustering gives five usable segments, not five natural groups.** K-Means with K=5 (chosen with Elbow, Silhouette, Davies-Bouldin, stability, and marketing usefulness) separates groups that need different actions. Silhouette is 0.318 and the dendrogram shows no single natural cluster count, so segment borders are soft. DBSCAN isolates 33 key accounts (0.8% of customers) that generate 21.5% of revenue.

**5. The segments map to specific actions.**

| Segment | Customers | Revenue share | Priority action |
|---|---|---|---|
| Champions | 8.3% | 53.4% | Protect: loyalty perks, account management for key accounts |
| Active Regulars | 22.5% | 23.7% | Grow: cross-sell, bundles, volume incentives |
| Cooling Regulars | 23.1% | 15.5% | Recover: targeted win-back, highest past spend first |
| Recent One-timers | 20.2% | 3.8% | Convert: welcome sequence aimed at a second order |
| Lost One-timers | 25.9% | 3.5% | Limit spend: one low-cost reactivation attempt |

### Bottom line

Champions and Active Regulars are 30.8% of customers but 77.1% of revenue, so protecting them is the first priority. The largest recoverable opportunity is Cooling Regulars, who showed repeat behavior and then went quiet. Marketing spend on single-order customers should depend on how recent they are.

### Caveats

The analysis is descriptive and based on one year of data (December 2011 incomplete) and registered customers only. Recommendations are hypotheses to validate with A/B tests, not proven effects (see Section 15).

## Save Segments

The final customer table with segment labels and outlier flags is saved for the Streamlit app.

In [15]:
SEGMENT_NAMES = {
    1: "Champions",
    4: "Active Regulars",
    2: "Recent One-timers",
    0: "Cooling Regulars",
    3: "Lost One-timers",
}

rfm["segment"] = rfm["kmeans"].map(SEGMENT_NAMES)
assert rfm["segment"].notna().all()

out = rfm.drop(columns=["log10_frequency", "log10_monetary"], errors="ignore")
out.to_parquet("../data/customer_segments.parquet", index=False)

check = pd.read_parquet("../data/customer_segments.parquet")
print("Saved customers:", len(check))
print("Columns:", list(check.columns))
print(check["segment"].value_counts().to_string())
print("Outliers:", int(check["is_outlier"].sum()))

Saved customers: 4320
Columns: ['CustomerID', 'frequency', 'gross_revenue', 'avg_order_value', 'avg_units_per_order', 'avg_lines_per_order', 'first_purchase', 'last_purchase', 'recency', 'tenure_days', 'cancelled_value', 'monetary', 'is_uk', 'R', 'F', 'M', 'RFM_Score', 'RFM_Sum', 'kmeans', 'dbscan', 'is_outlier', 'segment']
segment
Lost One-timers      1120
Cooling Regulars      997
Active Regulars       973
Recent One-timers     871
Champions             359
Outliers: 33
